# Team Ops Demo From Saved Index (Fast)

This notebook skips embedding creation and loads a previously saved index from Drive.
Use this for workshop demos to avoid slow embedding rebuilds.

In [ ]:
# Install dependencies
%pip install -q jedi llama-index llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
# Install system dependencies and Ollama
!apt-get -qq update && apt-get -qq install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama
import subprocess, time, os
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
print('Ollama started.')

In [ ]:
# Pull only what is needed for querying
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Load Persisted Index

Expected default path:
- `/content/drive/MyDrive/team_ops_index_nomic`

In [ ]:
from llama_index.core import Settings, StorageContext, load_index_from_storage
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from pathlib import Path

persist_dir = Path('/content/drive/MyDrive/team_ops_index_nomic')
if not persist_dir.exists():
    raise FileNotFoundError(f'Index folder not found: {persist_dir}')

Settings.llm = Ollama(model='llama3.2:3b', request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name='nomic-embed-text')

storage_context = StorageContext.from_defaults(persist_dir=str(persist_dir))
index = load_index_from_storage(storage_context)
query_engine = index.as_query_engine(similarity_top_k=4)

print('Loaded index from Drive.')

In [ ]:
# Plain LLM vs RAG
question = 'How do I submit my timesheets?'
plain = Settings.llm.complete(question)
rag = query_engine.query(question)

print('=== Plain LLM ===')
print(getattr(plain, 'text', str(plain)))
print('\n=== RAG ===')
print(rag)

In [ ]:
# Ask your own
q = input('Ask a question about Ops docs: ')
print(query_engine.query(q))